In [14]:
import mne
import mne_bids
from mne_bids import BIDSPath
import os, os.path as op
import nilearn
import matplotlib.pyplot as plt
import numpy as np
import glob
import nih2mne

from nilearn import *

from nih2mne.utilities.bids_helpers import get_mri_dict
n_jobs = 10 #Number of parrallel operations

## Set some generic BIDS information

In [19]:
#Raw data
bids_root = op.join('/data/', os.environ['USER'], 'meg_workshop_data')  
#Processed Data Folder: contains (freesurfer / Day1 / Day2) 
deriv_root = op.join(bids_root, 'derivatives')  
project_root = op.join(deriv_root, 'Workshop')  
#Freesurfer brain surface reconstruction
fs_subjects_dir = subjects_dir = op.join(deriv_root, 'freesurfer','subjects')

### Find data for a subject

In [20]:
# Let's loop over all the subjects and create a list of freesurfer subjects
subjects=['ON02811','ON03748','ON22671','ON42107','ON52662','ON61373','ON62003','ON70467','ON72082',
         'ON84896','ON85305','ON89474','ON89475']
fs_subjects=[]
for subject in subjects:
    fs_subjects.append('sub-' + subject)

#### Now, we are going to do exactly what we did in the MRI processing labs and the Source localization labs, but for every subject

In [ ]:
# declare a few empty lists to hold the source estimates

stcs_stim=[]
stcs_missing_stim=[]
stcs_contrast=[]

for subject in subjects:
    # filenames and load MRI files
    fs_subject = 'sub-' + subject
    bids_path = BIDSPath(root=bids_root, subject=subject, task='airpuff', run='01',session='01')
    project_path = bids_path.copy().update(root=project_root, check=False)
    
    data_dict = nih2mne.utilities.bids_helpers.get_mri_dict(subject,bids_root, task='airpuff')
    bem = data_dict['bem'].load()
    fwd = data_dict['fwd'].load()
    src = fwd['src']
    trans = data_dict['trans'].load()
    #Load the Raw data
    raw = mne.io.read_raw_ctf(bids_path.fpath, preload=True, verbose=False)
    bids_path = BIDSPath(root=bids_root, subject=subject, task='noise', run='01',session='01')
    noise = mne.io.read_raw_ctf(bids_path.fpath, preload=True, verbose=False)
    # filter for alpha        
    beta=raw.copy().filter(13,30)
    noise_beta=noise.copy().filter(13,30)
    evts, evtsid = mne.events_from_annotations(raw)
    epochs_beta = mne.Epochs(beta, evts, evtsid, tmin=-0.1, tmax=0.2, preload=True)
    cov_all_beta = mne.compute_covariance(epochs_beta, tmin=0.15, tmax = 0.25, n_jobs=n_jobs)
    cov_stim_beta = mne.compute_covariance(epochs_beta['stim'], tmin=.15, tmax = 0.25, n_jobs=n_jobs)
    cov_missingstim_beta = mne.compute_covariance(epochs_beta['missingstim'], tmin=0.15, tmax = 0.25, n_jobs=n_jobs)
    noise_cov_beta = mne.compute_raw_covariance(noise_beta, tmin=0, tmax=None, method="empirical", rank=None)
    filters_beta = mne.beamformer.make_lcmv(epochs_beta.info, fwd, cov_all_beta, reg=0.05, pick_ori='max-power',
                                   noise_cov = noise_cov_beta, weight_norm='unit-noise-gain')
    stc_stim_beta = mne.beamformer.apply_lcmv_cov(cov_stim_beta, filters_beta)
    stc_missing_beta = mne.beamformer.apply_lcmv_cov(cov_missingstim_beta, filters_beta)
    stc_contrast=stc_stim_beta.copy()
    stc_contrast.data=stc_stim_beta.data/stc_missing_alpha.data
    stcs_stim.append(stc_stim_beta)
    stcs_missing_stim.append(stc_missing_beta)
    stcs_contrast.append(stc_contrast)

Loading surfaces...

Loading the solution matrix...

Homogeneous model surface loaded.
Loaded linear collocation BEM solution from /data/nugenta/meg_workshop_data/derivatives/Workshop/sub-ON02811/ses-01/meg/sub-ON02811_ses-01_task-airpuff_run-01_bem.fif
Reading forward solution from /data/nugenta/meg_workshop_data/derivatives/Workshop/sub-ON02811/ses-01/meg/sub-ON02811_ses-01_task-airpuff_run-01_fwd.fif...
    Reading a source space...
    Computing patch statistics...
    Patch information added...
    Distance information added...
    [done]
    Reading a source space...
    Computing patch statistics...
    Patch information added...
    Distance information added...
    [done]
    2 source spaces read
    Desired named matrix (kind = 3523) not available
    Read MEG forward solution (8196 sources, 272 channels, free orientations)
    Source spaces transformed to the forward solution coordinate frame
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 13 - 30

[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.1s
[Parallel(n_jobs=1)]: Done  71 tasks      | elapsed:    0.6s
[Parallel(n_jobs=1)]: Done 161 tasks      | elapsed:    1.5s
[Parallel(n_jobs=1)]: Done 287 tasks      | elapsed:    2.7s


Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 13 - 30 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 13.00
- Lower transition bandwidth: 3.25 Hz (-6 dB cutoff frequency: 11.38 Hz)
- Upper passband edge: 30.00 Hz
- Upper transition bandwidth: 7.50 Hz (-6 dB cutoff frequency: 33.75 Hz)
- Filter length: 4875 samples (1.016 s)



[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.2s
[Parallel(n_jobs=1)]: Done  71 tasks      | elapsed:    1.0s
[Parallel(n_jobs=1)]: Done 161 tasks      | elapsed:    2.4s
[Parallel(n_jobs=1)]: Done 287 tasks      | elapsed:    4.3s


Used Annotations descriptions: ['missingstim', 'stim']
Not setting metadata
500 matching events found
Setting baseline interval to [-0.1, 0.0] s
Applying baseline correction (mode: mean)
0 projection items activated
Using data from preloaded Raw for 500 events and 361 original time points ...
0 bad epochs dropped
Removing 5 compensators from info because not all compensation channels were picked.
Computing rank from data with rank=None
    Using tolerance 7.9e-09 (2.2e-16 eps * 272 dim * 1.3e+05  max singular value)
    Estimated rank (mag): 272
    MAG: rank 272 computed from 272 data channels with 0 projectors
Reducing data rank from 272 -> 272
Estimating covariance using EMPIRICAL
Done.
Number of samples used : 30500
[done]
Removing 5 compensators from info because not all compensation channels were picked.
Computing rank from data with rank=None
    Using tolerance 7.2e-09 (2.2e-16 eps * 272 dim * 1.2e+05  max singular value)
    Estimated rank (mag): 272
    MAG: rank 272 computed

In [ ]:
## Note that I haven't done anything below here. I'm going to add in things from Lab5 and from this existing
## Lab. You don't need to do anything with that, I can continue it on Monday. I might try to do something 
## over the weekend as well.

In [ ]:
#Load the Raw data
bids_path = BIDSPath(root=bids_root, subject=subject, task='airpuff', run='01',session='01')
raw = mne.io.read_raw_ctf(bids_path.fpath, clean_names=True, preload=True, verbose=False)
raw.filter(0.3, 110, n_jobs=n_jobs)

### Prepare Imaging for Source Localization

In [ ]:
volfwd

In [ ]:
fs_volsrc_fname=op.join(project_root, 'fsave-src.fif')
fs_volsrc = mne.read_source_spaces(fs_volsrc_fname)

morph = mne.compute_source_morph(volfwd['src'], subject_from = 'sub-'+subject, 
                                 subject_to='fsaverage', subjects_dir=fs_subjects_dir, src_to=fs_volsrc)

### Load up one of the volume source estimate files for subject ON02811

In [ ]:
stc_fname = op.join(project_root, 'sub-ON02811', 'ses-01', 'sub-ON02811_ses-01_task-airpuff_run-01_desc-stim_stc30to50-vl.stc')
stc=mne.read_source_estimate(stc_fname)
stc

In [ ]:
stc_fsaverage = morph.apply(stc)

In [ ]:
_ = stc.plot(volsrc, subject='sub-'+subject, subjects_dir = fs_subjects_dir)

In [ ]:
fs_volsrc_fname=op.join(project_root, 'fsave-src.fif')
fs_volsrc = mne.read_source_spaces(fs_volsrc_fname)

In [ ]:
_ = stc_fsaverage.plot(fs_volsrc_fname, subject='fsaverage', subjects_dir = fs_subjects_dir)

In [ ]:
stc_list

In [ ]:
stc_list = glob.glob(op.join(project_root, 'sub-*', 'ses-01', 'sub-*_ses-01_task-airpuff_run-01_desc-stim_stc30to50-vl.stc'))

In [ ]:
volfwd

In [ ]:
stc_fsave_list = []
for stc_fname in stc_list:
    subject = op.basename(stc_fname).split('_')[0][4:]
    data_dict = get_mri_dict(subject,bids_root, task='airpuff', project='Day2')
    stc = mne.read_source_estimate(stc_fname)
    volfwd=data_dict['volfwd'].load()
    morph = mne.compute_source_morph(volfwd['src'], subject_from = 'sub-'+subject, 
                                 subject_to='fsaverage', subjects_dir=fs_subjects_dir, src_to=fs_volsrc)
    stc_fsave_list.append( morph.apply(stc ))
    
    

In [ ]:
#Convert to numpy array
group_dat = np.stack([stc_.data for stc_ in stc_fsave_list])

## Don't do statistics this way -- just a quick way to visualize

In [ ]:
import copy
stc_allav = copy.copy(stc_fsaverage)
#PUt the average of the numpy array into the new data
stc_allav._data = group_dat.mean(axis=0)

In [ ]:
_ = stc_allav.plot(fs_volsrc_fname, subject='fsaverage', subjects_dir = fs_subjects_dir)

## Parcellation

In [ ]:
labels=mne.read_labels_from_annot('sub-'+subject,parc='aparc', 
                                   subjects_dir=subjects_dir)        




In [ ]:
stc_surf_fname = op.join(project_root, 'TEST-src.fif')

In [ ]:
stc_surf = mne.read_source_estimate(stc_surf_fname)

In [ ]:
ts = mne.extract_label_time_course([stc_surf], labels, fwd['src'])[0]

## TS is a list (if epochs are entered) - We only have the average data here

In [ ]:
ts

In [ ]:
ts.shape

In [ ]:
len(labels)

### Because these are large parcels - the activity in the somatosensory cortex is reduced / averaged over teh rest of the parcel

In [ ]:
plt.plot(stc_surf.times, ts.T)